In [1]:
from rsl_rl.env import VecEnv
from rsl_rl.runners import OnPolicyRunner
import torch
import numpy as np
torch.set_printoptions(precision=5, linewidth=200, sci_mode=False)

N          = 100020
state_dim  = 8
action_dim = 5
T          = 60
budget     = 3600.
N_envs     = 1024 * 64
ΔT         = 1.0
device     = 'cuda'
usr_std = 0.01
test_num = 100
seed = 0

parameter_static = dict(N          = N,
                        B          = budget,
                        T          = T,
                        alpha      = 0.6,
                        v_max      = 0.2/14,
                        cost_ti    = 0.0977,
                        cost_ta    = 0.02,
                        cost_v     = 0.07,
                        cost_poc_0 = 0.000369,
                        cost_poc_1 = 0.001057,
                        pid        = 1.10/1000,
                        psr        = 0.7,
                        pid_plus   = 0.1221/1000,
                        pir        = 1/8,
                        prs        = 1/7)

Random_mean = {"beta2" : 0.78735,
               "beta1" : 0.15747,
               "pei"   : 0.10714,
               "per"   : 0.04545}

space = {"beta2" : [Random_mean['beta2'],Random_mean['beta2']*usr_std],
         "beta1" : [Random_mean['beta1'], Random_mean['beta1']*usr_std],
         "pei"   : [Random_mean['pei'], Random_mean['pei']*usr_std],
         "per"   : [Random_mean['per'], Random_mean['per']*usr_std]}

Init_state = torch.tensor([[100000 / N, 20 / N, 0, 0, 0, 0, 1, 0]], 
                         dtype=torch.float32).to(torch.device(device))

In [2]:
class myVecEnv(VecEnv):
    def __init__(self, interval, n_envs = N_envs):
        self.B = parameter_static['B']  # total budget
        self.N = parameter_static['N']
        self.device = device
        self.init_state = Init_state
        self.states = self.init_state.repeat(n_envs, 1)
        self.num_envs = n_envs
        self.num_obs = state_dim
        self.num_privileged_obs = None
        self.num_actions = action_dim
        self.max_episode_length = parameter_static['T']
        
        torch.manual_seed(seed)
        self.parameter_random = {
            key: torch.empty(self.num_envs).normal_(
                interval[key][0],
                interval[key][1]
            ).to(torch.device(device))
            for key, value in Random_mean.items()
        }

        for key, value in Random_mean.items():
            indice = self.parameter_random[key] < 0
            self.parameter_random[key][indice] = -self.parameter_random[key][indice] 

        # optimization flags for pytorch JIT
        torch._C._jit_set_profiling_mode(False)
        torch._C._jit_set_profiling_executor(False)
        
        # allocate buffers
        self.privileged_obs_buf = None
        self.obs_buf = torch.zeros(self.num_envs, self.num_obs, device=self.device, dtype=torch.float)
        self.rew_buf = torch.zeros(self.num_envs, device=self.device, dtype=torch.float)
        self.reset_buf = torch.ones(self.num_envs, device=self.device, dtype=torch.long)
        self.episode_length_buf = torch.zeros(self.num_envs, device=self.device, dtype=torch.long)
        
        self.actions = torch.zeros(self.num_envs, self.num_actions, dtype=torch.float, device=self.device, requires_grad=False)
        self.episode_sums = torch.zeros(self.num_envs, dtype=torch.float, device=self.device, requires_grad=False)
        self.episode_acts = torch.zeros(self.num_envs, dtype=torch.float, device=self.device, requires_grad=False)
        self.extras = {}
    
    def para_update(self, para):
        self.parameter_random = para
        
    def step(self, actions):
        # actions clip
        soft_bound = [-0.5, 1.5]
        out_of_limits = -(actions - soft_bound[0]).clip(max=0.)
        out_of_limits += (actions - soft_bound[1]).clip(min=0.)
        scale_action = -0.1
        reward_action = scale_action * torch.sum(out_of_limits, dim=1).clip(max=1.).to(self.device)
        self.actions = torch.clip(actions, 0., 1.).to(self.device)

        # update actions & states
        b, p = self.states[:, -2] * self.B, self.states[:, -1]
        inv_c = self.cost_function(self.actions)
        b -= inv_c
        b = torch.clip(b, 0., self.B).to(self.device)
        mask = (b > 0).float().unsqueeze(-1)
        self.act = self.actions * mask
        inv_c = self.cost_function(self.act)
        next_state, reward_original = self.transition(self.act)
        self.extras['reward_original'] = reward_original
        self.extras['inv_cost'] = inv_c
        self.episode_length_buf += 1
        p = self.episode_length_buf / self.max_episode_length
        self.states = torch.cat([next_state / self.N, (b / self.B).unsqueeze(-1), p.unsqueeze(-1)], dim=-1)
        
        # check termination
        self.reset_buf = self.episode_length_buf >= self.max_episode_length
        
        # compute reward
        r_mean = -6584
        r_std  =  7347
        reward = (reward_original - inv_c - r_mean) / r_std
        #if reward > 0: 
        #reward = torch.exp(reward)/10
        #reward = torch.exp((1e5 + reward_original - inv_c) / 5e4)
        self.rew_buf       = reward + reward_action
        self.episode_sums += reward + reward_action
        self.episode_acts += reward_action
        
        # compute resets
        env_ids = self.reset_buf.nonzero(as_tuple=False).flatten()
        self.reset_idx(env_ids)
        
        # compute observations
        return self.get_observations(), self.states, self.rew_buf, self.reset_buf, self.extras

    def reset_idx(self, env_ids):
        """Reset selected robots"""
        if len(env_ids) == 0:
            return
        
        assert len(env_ids) == self.num_envs
        # reset robot states
        self.states[env_ids] = self.init_state
        # reset buffers
        self.episode_length_buf[env_ids] = 0
        self.reset_buf[env_ids] = 1
        # fill extras
        self.extras["episode"] = {}
        self.extras["episode"]['rew_sum'] = torch.mean(self.episode_sums[env_ids]) / (self.max_episode_length * ΔT)
        self.extras["episode"]['rew_act'] = torch.mean(self.episode_acts[env_ids]) / (self.max_episode_length * ΔT)
        self.episode_sums[env_ids] = 0.
        self.episode_acts[env_ids] = 0.
    
    def reset(self):
        """ Reset all robots"""
        self.reset_idx(torch.arange(self.num_envs, device=self.device))
        return {}, {}
    
    def get_observations(self):
        o_mean = torch.tensor([[0.58712867, 0.06804276, 0.00308146, 0.25684301, 0.00261079, 0.08229331, 0.26572794, 0.51      ]], dtype=torch.float32).to(torch.device(self.device))
        o_std  = torch.tensor([[0.41484398, 0.07883003, 0.00389494, 0.292568  , 0.00394907, 0.06906768, 0.33943075, 0.28861739]], dtype=torch.float32).to(torch.device(self.device))
        self.obs_buf = (self.states - o_mean) / o_std
        return self.obs_buf
    
    def get_privileged_observations(self):
        return self.privileged_obs_buf

    def cost_function(self, action):
        states = self.states[:,:-2] * self.N
        st, et, at, it, dt, rt = states[:, 0], states[:, 1], states[:, 2], states[:, 3], states[:, 4], states[:, 5]
        u_sv, u_it, u_aq, u_iq, w = action[:, 0], action[:, 1], action[:, 2], action[:, 3], action[:, 4]

        cost_it  = it * u_it * parameter_static['cost_ti']
        cost_v   = st * u_sv * parameter_static['cost_v'] * parameter_static['v_max']
        cost_q   = (u_aq * at + u_iq * it) * parameter_static['cost_ta']
        per_poc  = ((st + et + at + rt) / self.N).pow(2) * parameter_static['cost_poc_0'] + parameter_static['cost_poc_1']
        cost_poc = (st + et + rt + at) * w * per_poc
        return cost_poc + cost_it + cost_v + cost_q

    def transition(self, action):
        states = self.states[:,:-2] * self.N
        st, et, at, it, dt, rt = states[:, 0], states[:, 1], states[:, 2], states[:, 3], states[:, 4], states[:, 5]
        u_sv, u_it, u_aq, u_iq, w = action[:, 0], action[:, 1], action[:, 2], action[:, 3], action[:, 4]
        ht = st + et + at + it + rt

        st_out   = (1 - u_sv * parameter_static['v_max']) * st * (it * (1 - u_iq) * self.parameter_random['beta1'] + (et + at * (1 - u_aq)) * self.parameter_random['beta2']) / ht
        st_out_e = st_out * parameter_static['alpha']
        st_out_i = st_out * (1 - parameter_static['alpha'])
        st_out_r = u_sv * parameter_static['v_max'] * st * parameter_static['psr']
        et_out_r = et * self.parameter_random['per']
        et_out_i = et * self.parameter_random['pei']
        et_out_a = et * w * (1 - self.parameter_random['per'] - self.parameter_random['pei'])
        at_out_i = at * self.parameter_random['pei']
        at_out_r = at * self.parameter_random['per']
        it_out_r = it * parameter_static['pir'] * u_it
        it_out_d = it * parameter_static['pid'] * (1 - u_it) + it * u_it * parameter_static['pid_plus']
        rt_out_s = rt * parameter_static['prs']
        
        stn = st - st_out_e - st_out_i - st_out_r + rt_out_s
        etn = et - et_out_r - et_out_i - et_out_a + st_out_e
        atn = at - at_out_r - at_out_i + et_out_a
        itn = it - it_out_r - it_out_d + st_out_i + et_out_i + at_out_i
        dtn = dt + it_out_d
        rtn = rt + et_out_r + at_out_r + it_out_r + st_out_r - rt_out_s

        reward = -2.6 * st_out - 100 * it_out_d

        next_state = torch.stack([stn, etn, atn, itn, dtn, rtn], dim=-1)
        return next_state, reward

In [3]:
import inspect

class BaseConfig:
    def __init__(self) -> None:
        """ Initializes all member classes recursively. Ignores all namse starting with '__' (buit-in methods)."""
        self.init_member_classes(self)
    
    @staticmethod
    def init_member_classes(obj):
        # iterate over all attributes names
        for key in dir(obj):
            # disregard builtin attributes
            # if key.startswith("__"):
            if key=="__class__":
                continue
            # get the corresponding attribute object
            var =  getattr(obj, key)
            # check if it the attribute is a class
            if inspect.isclass(var):
                # instantate the class
                i_var = var()
                # set the attribute to the instance instead of the type
                setattr(obj, key, i_var)
                # recursively init members of the attribute
                BaseConfig.init_member_classes(i_var)

class LeggedRobotCfgPPO(BaseConfig):
    seed = 1
    runner_class_name = 'OnPolicyRunner'
    class policy:
        init_noise_std = 2
        actor_hidden_dims  = [64, 32, 16] 
        critic_hidden_dims = [64, 32, 16]
        activation = 'elu' # can be elu, relu, selu, crelu, lrelu, tanh, sigmoid
        # only for 'ActorCriticRecurrent':
        # rnn_type = 'lstm'
        # rnn_hidden_size = 512
        # rnn_num_layers = 1
        
    class algorithm:
        # training params
        value_loss_coef = 0.5
        use_clipped_value_loss = True
        clip_param = 0.2
        entropy_coef = 0.01
        num_learning_epochs = 5
        num_mini_batches = 64 # mini batch size = num_envs*nsteps / nminibatches
        learning_rate = 1.e-4
        schedule = 'adaptive' # could be adaptive, fixed
        gamma = 0.99
        lam = 0.95
        desired_kl = 0.01
        max_grad_norm = 1.

    class runner:
        policy_class_name = 'ActorCritic'
        algorithm_class_name = 'PPO'
        num_steps_per_env =  150 # per iteration
        max_iterations    = 100 # number of policy updates

        # logging
        save_interval = 100 # check for potential saves every this many iterations
        experiment_name = 'test'
        run_name = ''
        # load and resume
        resume = True
        load_run = -1 # -1 = last run
        checkpoint = -1 # -1 = last saved model
        resume_path = None # updated from load_run and chkpt
        
def class_to_dict(obj) -> dict:
    if not  hasattr(obj,"__dict__"):
        return obj
    result = {}
    for key in dir(obj):
        if key.startswith("_"):
            continue
        element = []
        val = getattr(obj, key)
        if isinstance(val, list):
            for item in val:
                element.append(class_to_dict(item))
        else:
            element = class_to_dict(val)
        result[key] = element
    return result

In [4]:
def para_sampling(intvl, num_sample=N_envs):
    parameter_random = {
        key: torch.empty(num_sample).normal_(
        intvl[key][0],
        intvl[key][1]
        ).to(torch.device(device))
        for key, value in Random_mean.items()
    }
    for key, value in Random_mean.items():
        indice = parameter_random[key] < 0
        parameter_random[key][indice] = -parameter_random[key][indice] 
    return parameter_random

def update_space(final_indices, para_buf, space):
    para_est = {}
    for key, tensor_values in para_buf.items():
        selected_values = torch.mean(tensor_values[final_indices[0], final_indices[1]]) 
        para_values = torch.mean(tensor_values[final_indices[0], final_indices[1]], dim=0)
        update_values = space[key][0] + 0.5 * (selected_values - space[key][0])
        space[key] = [update_values , update_values  * 0.1]
        para_est[key] = para_values
    return space, para_est


def cal_err(para_value, env_para):
    gap = []
    for key, value in env_para.items():
        gap_temp = torch.mean(torch.abs(para_value[key] - env_para[key])/(env_para[key]))
        gap.append(gap_temp)
    return sum(gap)/len(gap) # 所有值都在范围内，返回 0
    
def state_estimator(env_sample, action_sim, state_sim, state_prior, space, iter): 
    env_sample.reset()
    env_sample.states = state_prior  
    para_buf, mse_buf, obs_buf, states_buf = {}, {}, {}, {}
    for i in range(iter):
        para = para_sampling(space, test_num)
        env_sample.para_update(para)
        obs_sample, states_sample, _, _, _ = env_sample.step(action_sim.detach())  
        mse = torch.mean((state_sim[:,2:4] - states_sample[:,2:4]) ** 2, dim=1) 
        if i == 0:
            mse_buf = mse.unsqueeze(0)
            obs_buf = obs_sample.unsqueeze(0)
            states_buf = states_sample.unsqueeze(0)
            for key, values in para.items():
                para_buf[key] = para[key].unsqueeze(0)
        else:
            mse_buf = torch.cat((mse_buf, mse.unsqueeze(0)), dim=0)
            obs_buf = torch.cat((obs_buf, obs_sample.unsqueeze(0)),dim=0)
            states_buf = torch.cat((states_buf, states_sample.unsqueeze(0)), dim=0)
            for key, values in para.items():
                para_buf[key] = torch.cat((para_buf[key], para[key].unsqueeze(0)), dim=0)
                
     
    k = int(iter * 0.05)
    _, row_indices = torch.topk(mse_buf, k=k, dim=0, largest=False)
    column_indices = torch.arange(mse_buf.shape[1]).repeat(k, 1)
    top_indices = [row_indices, column_indices]
    obs_est = torch.mean(obs_buf[row_indices,column_indices,:], dim=0)
    state_est = torch.mean(states_buf[row_indices,column_indices,:], dim=0)
    state_est[:,-1] = state_prior[:,-1] + torch.tensor(1/T)
    return obs_est, state_est, top_indices, para_buf

In [5]:
para_err = 0.5
torch.manual_seed(seed)
para_true = []
para_temp = {
    key: torch.empty(test_num).uniform_(
        value * (1-para_err),
        value * (1+para_err)
    ).to(torch.device(device))
    for key, value in Random_mean.items()
}


para = {}
for t in range(T):
    para = {
        key: (para_temp[key] - para_temp[key] * torch.sin(torch.tensor(t/(3.1415926 * 3))) * 0.25).to(torch.device(device))
        for key, value in Random_mean.items()
        }
    para_true.append(para)


In [6]:
env = myVecEnv(space, test_num)
env.para_update(para_true[0])
train_cfg = LeggedRobotCfgPPO()
train_cfg_dict = class_to_dict(train_cfg)


env_transfer = myVecEnv(space)
runner = OnPolicyRunner(env_transfer, train_cfg_dict, f'rsl_test', device=env.device)
runner.load(f'rsl_output_main/model_100.pt')

policy = runner.get_inference_policy(device=env.device)

reward_history = []
act_buf = []
state_his_buf = []
death_buf = []

env_sample = myVecEnv(space, test_num)
state_est = env.states
obs_est = env.get_observations()
for t in range(T-1):
    actions = policy(obs_est.detach())
    obs, states, _, _, infos = env.step(actions.detach())
    reward_history.append((-infos['reward_original'] + infos['inv_cost']).tolist())
    death_buf.append((-infos['reward_original']).tolist())
    act_buf.append(env.act)
    state_his_buf.append(env.states)
    obs_est, state_buf, top_indices, para_buf = state_estimator(env_sample, actions, states, state_est, space, 1024)
    space, para_v = update_space(top_indices, para_buf, space) 

    runner.learn(num_learning_iterations=25, init_at_random_ep_len=False)
    policy = runner.get_inference_policy(device=env.device)
    env_transfer = myVecEnv(space)
    runner = OnPolicyRunner(env_transfer, train_cfg_dict, f'rsl_test', device=env.device)
    runner.load(f'rsl_test/model_{int(t)* 25}.pt')
        

    if t < T - 1: env.para_update(para_true[t + 1])
    print(f'score.mean: {np.sum(reward_history, axis=0).mean()/T}')
env.reset()

reward_history = np.array(reward_history)
print(f'score     : {np.sum(reward_history, axis=0)/T       }')
print(f'score.mean: {np.sum(reward_history, axis=0).mean()/T}')

Actor MLP: Sequential(
  (0): Linear(in_features=8, out_features=64, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=32, out_features=16, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=16, out_features=5, bias=True)
)
Critic MLP: Sequential(
  (0): Linear(in_features=8, out_features=64, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=32, out_features=16, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=16, out_features=1, bias=True)
)
################################################################################
                        Learning iteration 0/25                         

                       Computation: 2407005 steps/s (collection: 0.634s, learning 3.450s)
               Value function loss: 142.5378
                    Surrogate loss: 0.0002
             Mean action

In [7]:
a = np.sum(reward_history, axis=0)/T 

In [8]:
min(a)

34.21917556989938

In [9]:
max(a)

5468.090394965808

In [10]:
np.std(a)

1469.501008861074

In [11]:
exp_result = {
    'reward_total': reward_history,
    'reward_peo': death_buf,
    'state': state_his_buf,
    'action': act_buf
}

In [12]:
torch.save(exp_result, 'mid_budget_result.pt')